# Week 7 — Euler vs Hamilton on a tiny genome

Open in VS Code. **Select Kernel** (Python 3) in the top right. Then **Run All** from the top.

The code already runs. Read the comments, change one input if you want, watch the printout. Then answer the questions.

Example 1: the genome $\mathtt{ACAAT}$, $k=3$. Example 2: $\mathtt{AAACAA}$, which contains a repeat.

In [ ]:
def kmers(s, k):
    return [s[i : i + k] for i in range(len(s) - k + 1)]


S, k = "ACAAT", 3
print("k-mers", kmers(S, k))


## Overlap graph

Vertices = $k$-mers. Edge $u\to v$ if the suffix of $u$ equals the prefix of $v$.


In [ ]:
ks = kmers(S, k)
print("vertices", ks)
print("edges:")
for u in ks:
    for v in ks:
        if u[1:] == v[:-1]:
            print(" ", u, "->", v)


## de Bruijn

Vertices = $(k-1)$-mers, edges = $k$-mers. An Eulerian path spells the string.


In [ ]:
nodes = sorted({m[:-1] for m in ks} | {m[1:] for m in ks})
print("nodes", nodes)
print("edges (k-mer : from -> to):")
for m in ks:
    print(" ", m, ":", m[:-1], "->", m[1:])

# Play: a repeat. k=3, S2 has two ways to go after "AA".
S2 = "AAACAA"
print("\nrepeat string", S2, "k-mers", kmers(S2, 3))
for m in kmers(S2, 3):
    print(" ", m, ":", m[:-1], "->", m[1:])


## Layout size: overlap vs de Bruijn

Vertices of the overlap graph = $k$-mers (one per read window). Vertices of de Bruijn = distinct $(k-1)$-mers. Watch which $|V|$ stays small as the string grows.


In [ ]:
def overlap_size(s, k):
    ks = kmers(s, k)
    edges = sum(1 for u in ks for v in ks if u[1:] == v[:-1])
    return len(ks), edges


def dbg_size(s, k):
    ks = kmers(s, k)
    nodes = {m[:-1] for m in ks} | {m[1:] for m in ks}
    return len(nodes), len(ks)


for s in ("ACAAT", "ACAATACAAT", "A" * 30):
    ov, db = overlap_size(s, 3), dbg_size(s, 3)
    print(s[:20] + ("..." if len(s) > 20 else ""), " overlap |V|,|E|", ov, "  dbg |V|,|E|", db)


## Degrees decide whether a walk exists

Count arrows in and out of every vertex. A path needs one start (out = in + 1) and one end (in = out + 1); a circuit needs every vertex balanced.

In [ ]:
from collections import defaultdict


def debruijn_edges(kmer_list):
    """Each k-mer is an arrow from its first k-1 letters to its last k-1 letters."""
    return [(m[:-1], m[1:]) for m in kmer_list]


def degrees(edges):
    d_in, d_out = defaultdict(int), defaultdict(int)
    for u, v in edges:
        d_out[u] += 1
        d_in[v] += 1
    for x in sorted(set(d_in) | set(d_out)):
        print(f"  {x}: in {d_in[x]}  out {d_out[x]}")
    return d_in, d_out


print("example 1:")
E1 = debruijn_edges(kmers("ACAAT", 3))
degrees(E1)

## Hierholzer's algorithm

Walk unused edges until stuck; if edges remain, start a loop from a vertex on the walk and splice it in. Each edge is used once.

In [ ]:
def hierholzer(edges, start):
    out = defaultdict(list)
    for u, v in edges:
        out[u].append(v)
    stack, walk = [start], []
    while stack:
        u = stack[-1]
        if out[u]:
            stack.append(out[u].pop(0))   # follow an unused edge
        else:
            walk.append(stack.pop())      # stuck: this vertex is finished
    return walk[::-1]


def spell(walk):
    return walk[0] + "".join(v[-1] for v in walk[1:])


w = hierholzer(E1, "AC")
print("example 1 walk:", " -> ".join(w), "  spells", spell(w))

# The bow-tie from the slides: A->B->C->A and B->D->E->B share B.
bowtie = [("A", "B"), ("B", "C"), ("C", "A"), ("B", "D"), ("D", "E"), ("E", "B")]
print("bow-tie walk:  ", " -> ".join(hierholzer(bowtie, "A")))

## Example 2: a repeat gives two genomes

$\mathtt{AAACAA}$ has the same $k$-mers as another genome. List every Eulerian circuit that starts at $\mathtt{AA}$.

In [ ]:
def all_circuits(edges, start):
    """Every walk from start that uses each edge exactly once (fine for tiny graphs)."""
    found = []

    def extend(walk, unused):
        if not unused:
            found.append(walk)
            return
        for i, (u, v) in enumerate(unused):
            if u == walk[-1]:
                extend(walk + [v], unused[:i] + unused[i + 1:])

    extend([start], list(edges))
    return found


E2 = debruijn_edges(kmers("AAACAA", 3))
degrees(E2)
for walk in all_circuits(E2, "AA"):
    print(" -> ".join(walk), "  spells", spell(walk))

## A larger $k$ resolves the repeat

The same genome with $k=4$: no 3-mer occurs twice, so there is only one way through.

In [ ]:
E2k4 = debruijn_edges(kmers("AAACAA", 4))
degrees(E2k4)
start = "AAA"   # out = in + 1
w = hierholzer(E2k4, start)
print("walk:", " -> ".join(w), "  spells", spell(w))

## Counting $k$-mers in reads with errors

Simulate reads from a random genome, count every $k$-mer in a dictionary (a hash table), and look at how often each count occurs: the $k$-mer spectrum.

In [ ]:
import random
from collections import Counter

import matplotlib.pyplot as plt

random.seed(2026)
L = 20000                          # genome length
G = "".join(random.choice("ACGT") for _ in range(L))
read_len, coverage, error, k = 100, 30, 0.01, 21

counts = Counter()
for _ in range(coverage * L // read_len):
    start = random.randrange(0, L - read_len + 1)
    read = list(G[start:start + read_len])
    for i in range(read_len):            # sequencing errors
        if random.random() < error:
            read[i] = random.choice([b for b in "ACGT" if b != read[i]])
    read = "".join(read)
    for i in range(read_len - k + 1):    # count every k-mer of the read
        counts[read[i:i + k]] += 1

spectrum = Counter(counts.values())
print("distinct k-mers in the reads:", len(counts), "   in the genome:", L - k + 1)
print("k-mers seen once:", spectrum[1])

xs = range(1, 41)
plt.figure(figsize=(7, 3))
plt.bar(xs, [spectrum.get(c, 0) for c in xs], color=["#b56a3a" if c <= 3 else "#2a5a8c" for c in xs])
plt.ylim(0, 2500)
plt.xlabel("times a k-mer occurs in the reads")
plt.ylabel("distinct k-mers")
plt.tight_layout()
plt.show()

peak = 19
solid = sum(c * n for c, n in spectrum.items() if c >= 5)
print(f"solid k-mer occurrences {solid},  genome size estimate {solid / peak:.0f}  (true {L})")

## Counting by hand

Four reads of `ATGCTCAGG` plus one read with a misread letter. Count the 3-mers, then count the counts.

In [ ]:
G_toy = "ATGCTCAGG"
reads_toy = [G_toy[i:i + 6] for i in range(len(G_toy) - 5)] + ["TGTTCA"]
c3 = Counter(r[i:i + 3] for r in reads_toy for i in range(len(r) - 2))
print(c3)
print("counts of counts:", sorted(Counter(c3.values()).items()))   # [(1, 5), (2, 2), (3, 1), (4, 2)]

## The price of a larger $k$

For every $k$-mer of the genome, count the reads that contain it **without an error**, and compare with the formula $\lambda(k) = \frac{c}{L}(L-k+1)(1-e)^k$. A $k$-mer with no error-free copy is missing from the graph.

In [ ]:
random.seed(2026)
G = "".join(random.choice("ACGT") for _ in range(L))
runs = []                                   # (start, error positions) for every read
for _ in range(coverage * L // read_len):
    start = random.randrange(0, L - read_len + 1)
    bad = []
    for i in range(read_len):
        if random.random() < error:
            random.choice("ACG")            # same random draws as above
            bad.append(i)
    runs.append((start, bad))

for kk in (21, 41, 61, 81):
    ok = [0] * L
    for start, bad in runs:
        for i in range(read_len - kk + 1):
            if not any(i <= b < i + kk for b in bad):
                ok[start + i] += 1
    inner = range(read_len, L - read_len)  # away from the genome's ends
    mean = sum(ok[j] for j in inner) / len(inner)
    lost = sum(ok[j] == 0 for j in inner) / len(inner)
    lam = coverage / read_len * (read_len - kk + 1) * (1 - error) ** kk
    print(f"k={kk}: formula {lam:.1f}, simulation {mean:.1f}, missing {100 * lost:.1f}%")

## An error creates a bubble

Reads of `ATGCTCAGG` (length 6) plus one read with a wrong letter. Count the 3-mers: the error path is seen once.

In [ ]:
G = "ATGCTCAGG"
reads = [G[i:i + 6] for i in range(len(G) - 5)] + ["TGTTCA"]   # last read: C misread as T
c3 = Counter(r[i:i + 3] for r in reads for i in range(len(r) - 2))
for kmer, c in sorted(c3.items()):
    print(f"  {kmer}: {kmer[:2]} -> {kmer[1:]}   seen {c}")

## Contigs

Merge every chain of vertices with one arrow in and one arrow out. Each merged chain is a contig.

In [ ]:
def contigs(kmer_list):
    edges = debruijn_edges(sorted(set(kmer_list)))
    out, d_in = defaultdict(list), defaultdict(int)
    for u, v in edges:
        out[u].append(v)
        d_in[v] += 1
    nodes = set(out) | set(d_in)
    inside = {v for v in nodes if d_in[v] == 1 and len(out[v]) == 1}
    pieces = []
    for v in nodes:
        if v in inside:
            continue
        for w in out[v]:              # start a contig at every branch or end
            path = [v, w]
            while w in inside:
                w = out[w][0]
                path.append(w)
            pieces.append(spell(path))
    return pieces


print(contigs(kmers("TTCCGGACCTCGGATGT", 4)))

## Hanoi: the transcript is the cost

7 lines for $n=3$. The algorithm is three lines; $n=4$ is 15 lines. Do not print $n=20$.


In [ ]:
def hanoi(n, src, dst, aux):
    if n == 0:
        return
    hanoi(n - 1, src, aux, dst)
    print(src, "->", dst)
    hanoi(n - 1, aux, dst, src)


hanoi(3, "start", "goal", "aux")


## Hanoi as a graph

A vertex is a word: for each disk, largest first, the peg it sits on (`S`, `A`, `G`). An edge is one legal move. Count vertices and edges, check the degrees, and find the shortest solution by breadth-first search.

In [ ]:
import itertools
from collections import deque

def hanoi_graph(n):
    adj = {"".join(w): set() for w in itertools.product("SAG", repeat=n)}
    for v in adj:
        for a in "SAG":
            on_a = [i for i in range(n) if v[i] == a]
            if not on_a:
                continue
            d = on_a[-1]                         # smallest disk on peg a is on top
            for b in "SAG":
                on_b = [i for i in range(n) if v[i] == b]
                if b != a and not (on_b and on_b[-1] > d):
                    adj[v].add(v[:d] + b + v[d + 1:])
    return adj

for n in range(1, 7):
    adj = hanoi_graph(n)
    edges = sum(len(w) for w in adj.values()) // 2
    degrees = Counter(len(w) for w in adj.values())
    dist = {"S" * n: 0}
    queue = deque(["S" * n])
    while queue:
        u = queue.popleft()
        for w in adj[u]:
            if w not in dist:
                dist[w] = dist[u] + 1
                queue.append(w)
    print(f"n={n}: {len(adj)} vertices, {edges} edges, degrees {dict(degrees)}, shortest solution {dist['G' * n]}")

In [ ]:
def every_arrangement(n, a, b):
    """Visit every arrangement once, from all on a to all on b: copy a, copy c, copy b."""
    c = next(q for q in "SAG" if q not in (a, b))
    if n == 1:
        return [a, c, b]
    return ([a + w for w in every_arrangement(n - 1, a, b)]
            + [c + w for w in every_arrangement(n - 1, b, a)]
            + [b + w for w in every_arrangement(n - 1, a, b)])

walk = every_arrangement(3, "S", "G")
adj = hanoi_graph(3)
print(len(walk), len(set(walk)), all(w in adj[v] for v, w in zip(walk, walk[1:])))   # 27 27 True

## Questions

Without looking at notes. **Q1–Q2:** today’s idea. **Q3:** a situation we did not practise.

**Q1.** Which path problem has a fast algorithm, Eulerian or Hamiltonian? How do the degrees decide whether a walk exists?

A1.

**Q2.** In the overlap graph and in the de Bruijn graph, what are the vertices and the edges? Which search is Hamiltonian, which Eulerian?

A2.

**Q3.** A plasmid is a circular DNA molecule (week 5). If all reads come from a plasmid, what do the degrees of its de Bruijn graph look like, and where does the walk start?

A3.